In [ ]:
"""

Description:
An open-source document-based Q&A assistant using LangChain, FAISS, HuggingFace Embeddings, and TinyLLaMA (1.1B) LLM.
Supports PDF and TXT files with chunking, embedding, and semantic retrieval.

Features:
- Load PDF/TXT documents
- Chunk with overlap
- Embed using all-MiniLM-L6-v2
- Store in FAISS vector store
- Query using TinyLLaMA model

"""

: 

In [ ]:
import os
from langchain.document_loaders import PyPDFLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

# Load a PDF or TXT file
def load_file(file_path):
    if file_path.lower().endswith(".pdf"):
        loader = PyPDFLoader(file_path)
    elif file_path.lower().endswith(".txt"):
        loader = TextLoader(file_path, encoding="utf-8", autodetect_encoding=True)
    else:
        raise ValueError("Unsupported file type. Please provide a .pdf or .txt file.")
    return loader.load()

# Main flow
def main():
    file_path = input("Enter path to PDF or TXT file: ").strip()

    print("\nLoading document...")
    docs = load_file(file_path)

    print("Splitting text into chunks...")
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_documents(docs)

    print("Generating embeddings using HuggingFace...")
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = FAISS.from_documents(chunks, embeddings)

    print("Loading TinyLLaMA model (open-source)...")
    model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    device = 0 if torch.cuda.is_available() else -1
    hf_pipeline = pipeline(
        "text-generation",
        model=model_id,
        tokenizer=model_id,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        device=device,
    )
    llm = HuggingFacePipeline(pipeline=hf_pipeline)

    qa = RetrievalQA.from_chain_type(llm=llm, retriever=vectorstore.as_retriever())

    print("\nReady! Ask your questions below.\n(Type 'exit' to quit.)")

    while True:
        query = input("Question: ")
        if query.lower() == "exit":
            break
        answer = qa.invoke({"query": query})
        print("Answer:", answer["result"])

if __name__ == "__main__":
    main()
